# [CITY] ROTBOTS -- Archive Scraper

---

Downloads and segments footage from archive.org. Skipped if ARCHIVE_RATIO = 0.

> **[?]** Found footage: who chose it? You or the machine?

---

In [ ]:
# SETUP
!pip install -q yt-dlp
import json, re, subprocess
from pathlib import Path
from IPython.display import display, Markdown
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
print('[OK] Setup')

In [ ]:
# LOAD SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'video_plan.json').exists()])
if not sessions: print('[!!] No plan! Run 01_Video_Plan first.')
else:
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
with open(SESSION_DIR/'video_plan.json') as f: plan = json.load(f)
NUM_ARCHIVE_SCENES = plan.get('num_archive_scenes', 0)
print(f'[OK] Session: {SESSION_NAME} | Archive scenes needed: {NUM_ARCHIVE_SCENES}')

---
## [CITY] Archive Sources

Add archive.org URLs or item IDs below.

In [ ]:
# ARCHIVE CONFIG
ARCHIVE_URLS = [
    # 'https://archive.org/details/example_item',
]

PREFER_HEIGHT = 480
MAX_CLIP_DURATION = 5  # match SECONDS_PER_SCENE
MIN_CLIP_DURATION = 3
MAX_EXTRACT_DURATION = 180

In [ ]:
# SCRAPE
def parse_archive_id(url):
    url = url.strip().rstrip('/')
    m = re.search(r'archive\.org/(?:details|download)/([^/?#]+)', url)
    if m: return m.group(1)
    if '/' not in url and '.' not in url: return url
    raise ValueError('Cannot parse: ' + url)

def get_dur(path):
    try: return float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(path)], capture_output=True, text=True, timeout=10).stdout.strip())
    except: return 0

archive_clips = []
ARCHIVE_DIR = SESSION_DIR / 'archive_clips'; ARCHIVE_DIR.mkdir(exist_ok=True)
if ARCHIVE_URLS and NUM_ARCHIVE_SCENES > 0:
    ATEMP = Path('/content/temp_archive'); ATEMP.mkdir(exist_ok=True)
    for i, url in enumerate(ARCHIVE_URLS):
        try: aid = parse_archive_id(url)
        except ValueError as e: print(f'Error: {e}'); continue
        print(f'Downloading [{i+1}/{len(ARCHIVE_URLS)}] {aid}')
        out_tpl = str(ATEMP / (aid + '.%(ext)s'))
        try: subprocess.run(['yt-dlp', 'https://archive.org/details/' + aid, '-f', 'bestvideo[height<=' + str(PREFER_HEIGHT) + ']+bestaudio/best[height<=' + str(PREFER_HEIGHT) + ']/best', '--merge-output-format', 'mp4', '--no-playlist', '--no-warnings', '-o', out_tpl, '--quiet'], check=True, timeout=600)
        except: print('  Download failed'); continue
        video = None
        for fp in ATEMP.iterdir():
            if fp.stem == aid and fp.suffix in ('.mp4','.mkv','.webm'): video = fp; break
        if not video: continue
        total = get_dur(video); clip_dir = ARCHIVE_DIR / aid; clip_dir.mkdir(exist_ok=True)
        extract_end = min(total, MAX_EXTRACT_DURATION) if MAX_EXTRACT_DURATION > 0 else total
        t = 0; idx2 = 0
        while t < extract_end:
            clip_dur = min(MAX_CLIP_DURATION, extract_end - t)
            if clip_dur < MIN_CLIP_DURATION: break
            clip_out = clip_dir / ('archive_' + f'{idx2:03d}' + '.mp4')
            r = subprocess.run(['ffmpeg','-y','-ss',str(t),'-i',str(video),'-t',str(clip_dur),'-c:v','libx264','-preset','fast','-crf','23','-an',str(clip_out)], capture_output=True, text=True, timeout=120)
            if r.returncode == 0 and clip_out.exists():
                archive_clips.append(dict(path=str(clip_out), duration=round(clip_dur,1), archive_id=aid)); idx2 += 1
            t += clip_dur
        video.unlink(missing_ok=True); print(f'  {idx2} clips')
    with open(SESSION_DIR / 'archive_clips.json', 'w') as f: json.dump(dict(clips=archive_clips, total=len(archive_clips)), f, indent=2)
    print(f'[OK] {len(archive_clips)} archive clips')
elif NUM_ARCHIVE_SCENES > 0: print(f' Need {NUM_ARCHIVE_SCENES} archive clips but no ARCHIVE_URLS!')
else: print('No archive footage needed')

---
*ROTBOTS Workshop -- LI-MA 2026*